# 이미지 압축과 설명 · 터미널 스크립트 작성 · v1.03

**2일차 6교시 · 수업 50분 + 휴식·전환 10분**

**목표:** main(image_path)에서 이미지를 압축해 Gemma 4로 보내고 설명을 터미널에 출력합니다.

슬라이드 설명 후 아래 순서대로 실습합니다. 커널은 **Python (Day2 LangChain)**입니다. API 표에서 입력·반환값을 먼저 확인하세요. 다른 노트북의 변수는 자동으로 공유되지 않습니다.

[수업 PDF](../슬라이드/강의자료_2일차_v2.pdf#page=41)

## 1. 파일 → JPEG 압축 → 전송 → 설명

이번 교시는 독립 Python 스크립트를 작성합니다. `main(image_path)`가 입력 경로 하나를 받아 전체 과정을 실행합니다. 이미지를 여는 코드는 main 내부에 둡니다. 학습 목표는 이미지 입력과 전송 흐름이며 JSON 판단 스키마는 필수가 아닙니다.

긴 변을 768픽셀 이하로 줄인 뒤 JPEG quality=80으로 저장합니다. 해상도와 품질을 낮추면 세부 글자가 사라질 수 있습니다. Base64는 압축이 아니며 JPEG 바이트를 전송용 문자열로 바꾸는 단계입니다. 작은 PNG에서는 결과 JPEG가 더 클 수도 있으므로 실제 바이트 수를 확인합니다.

설명·API 15분, main 구현 25분, 터미널 실행·원본 비교 10분.

![처리 흐름](assets/image-script-flow.svg)


In [ ]:
from pathlib import Path
import json
import os
import sys
from IPython.display import display, Image

# 현재 폴더에 교재가 없으면 부모 폴더를 차례로 확인합니다.
ROOT = Path.cwd().resolve()
while not (ROOT / "04.LLM_v2").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError("NEPES 교재 폴더 안에서 노트북을 여세요.")
    ROOT = ROOT.parent
# import가 교재의 공통 모듈을 찾을 수 있도록 경로를 등록합니다.
sys.path.insert(0, str(ROOT / "04.LLM_v2"))
sys.path.insert(0, str(ROOT / "05.VLM_v2"))
print("교재 위치:", ROOT)
print("현재 Python:", sys.executable)


In [ ]:
image_path = ROOT / "Data/images/05.VLM/road.png"
display(Image(filename=str(image_path)))
# 먼저 사람이 원본에서 직접 보이는 사실 두 가지를 적습니다.

### 사용 전 API 안내 · 스크립트에서 사용할 API

| 함수·클래스 | 초기화·사용 | 입력·반환값과 주의점 |
|---|---|---|
| `Image.open`, `ImageOps.exif_transpose` | `with Image.open(path) as original:` | 파일을 열고 EXIF 방향을 보정합니다. 파일 핸들은 with 종료 시 닫힙니다. |
| `convert`, `thumbnail` | `image.convert("RGB")`; `image.thumbnail((768, 768))` | JPEG에 맞는 색상 모드로 바꾸고 종횡비를 유지해 축소합니다. thumbnail은 객체를 직접 수정합니다. |
| `BytesIO`, `image.save` | `buffer = BytesIO()`; `image.save(buffer, format="JPEG", quality=80)` | 메모리 버퍼에 JPEG를 압축합니다. getvalue()는 압축 바이트를 반환합니다. |
| `base64.b64encode` | `base64.b64encode(compressed).decode("ascii")` | 바이트 → 전송 문자열. MIME은 image/jpeg입니다. |
| `ChatOllama`, `HumanMessage`, `invoke` | `ChatOllama(model="gemma4:e2b-it-qat", ...)` | 이미지·질문 블록을 메시지에 넣습니다. invoke에서 서버 요청, reply.content에서 답을 읽습니다. |
| `argparse.ArgumentParser` | `parser.add_argument("image_path")`; `parser.parse_args()` | 터미널의 위치 인자를 읽어 main에 넘깁니다. |

## 2. TODO · main(image_path) 구현

아래 main의 세 단계를 완성한 코드를 `05.VLM_v2/practice/describe_image.py`에 저장하세요. 해당 파일에는 import와 터미널 진입 코드가 준비되어 있습니다. 노트북의 함수만 고쳐서는 외부 스크립트가 변경되지 않습니다.

1. 이미지 열기·방향 보정·RGB 변환·해상도 축소
2. 메모리 JPEG 압축·Base64 data URI 생성
3. Gemma 4 요청·터미널 출력

실패한 파일 경로나 지원하지 않는 이미지 오류는 원문을 읽고 수정합니다. 원본 이미지는 덮어쓰지 않습니다.

In [ ]:
# 작성할 함수의 골격입니다. 실제 제출·실행 파일에도 완성한 내용을 저장하세요.
def main(image_path):
    # TODO: 이미지 열기 → JPEG 압축 → Gemma 4 요청 → print
    raise NotImplementedError("main의 세 단계를 완성하세요.")

In [ ]:
# 참고 정답: 아래는 파일 전체 내용입니다. 노트북에서는 함수 정의만 실행합니다.
"""이미지 한 장을 JPEG로 압축해 Gemma 4에 설명을 요청합니다."""
import argparse
import base64
from io import BytesIO
import os
from pathlib import Path
from PIL import Image, ImageOps
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama


def main_solution(image_path):
    # Path는 경로 문자열을 파일 경로 객체로 바꿉니다.
    path = Path(image_path)
    # with 블록이 끝나면 원본 파일을 닫습니다. 원본 파일은 덮어쓰지 않습니다.
    with Image.open(path) as original:
        image = ImageOps.exif_transpose(original).convert("RGB")
        original_size = image.size
        # 긴 변을 768 이하로 줄이고 종횡비를 유지합니다. 작은 이미지는 확대하지 않습니다.
        image.thumbnail((768, 768))
        buffer = BytesIO()  # JPEG 결과를 디스크 대신 메모리에 보관합니다.
        image.save(buffer, format="JPEG", quality=80, optimize=True)
        compressed = buffer.getvalue()
        resized_size = image.size
    # JPEG 압축은 화질·크기를 바꾸고, Base64는 전송할 문자 표현을 만듭니다.
    encoded = base64.b64encode(compressed).decode("ascii")
    data_uri = "data:image/jpeg;base64," + encoded
    model = ChatOllama(
        model="gemma4:e2b-it-qat",
        base_url=os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434"),
        temperature=0, reasoning=False, num_ctx=4096, num_predict=384,
        client_kwargs={"timeout": 180},
    )
    message = HumanMessage(content=[
        {"type": "text", "text": "이미지에서 직접 보이는 내용을 한국어로 두 문장으로 설명하세요."},
        {"type": "image_url", "image_url": data_uri},
    ])
    print(f"이미지: {original_size} → {resized_size}, JPEG {len(compressed):,} bytes", flush=True)
    print("Gemma 4 응답을 기다립니다...", flush=True)
    reply = model.invoke([message])
    if reply.response_metadata.get("done_reason") == "length":
        print("주의: 출력 한도로 답변이 잘렸습니다.")
    print(reply.content)  # 최종 설명을 터미널에 출력합니다.

### 터미널 진입 코드

파일의 함수 이름은 `main`으로 유지하고 아래 코드를 맨 끝에 둡니다. 노트북 참고 정답 함수 이름만 `main_solution`으로 구분했습니다.

```python
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("image_path", help="설명할 이미지 파일 경로")
    args = parser.parse_args()
    main(args.image_path)
```

## 3. 터미널에서 실행

**아래 스크립트 실행은 모델 로딩과 응답에 시간이 걸립니다.** Jupyter Terminal에서 입력하세요.

```bash
cd /workspace
Data/venvs/day2-langchain/bin/python 05.VLM_v2/practice/describe_image.py Data/images/05.VLM/road.png
```

파일 전체 정답은 `05.VLM_v2/practice/solutions/describe_image.py`에도 있습니다. 학생 스크립트와 같은 이미지로 비교할 수 있습니다. 아래 검증 셀은 노트북에서 작성한 main을 확인하며 외부 파일 저장 여부는 터미널 실행으로 확인합니다.

In [ ]:
active_main = main
if os.environ.get("DAY2_TODO_VARIANT") == "solution":
    active_main = main_solution
active_main(image_path)  # 실제 모델 호출 1회. 설명과 원본을 비교하세요.

## 4. 완료 확인

- 원본 파일을 바꾸지 않고 메모리에서 JPEG를 만들었나요?
- 이미지 설명이 터미널에 출력되나요?
- JPEG 압축과 Base64를 구분할 수 있나요?
- 해상도를 더 줄였을 때 놓치는 세부 정보가 있나요?

한 장의 이미지로 보이지 않는 원인·전후 움직임을 단정하지 않는지도 확인합니다.